# Nonlinearity-preserving quantum simulation of flows in the early fault-tolerant regime

This notebook follows the main figures of the accompanying paper, from a reduced cylinder-wake model to quantum circuits and fault-tolerant resource estimates. Short examples show how to fit a model, synthesize rotations, estimate resources for a user-supplied circuit, and select a measurement budget.

Run the cells in order after following the [installation and data instructions](README.md). The demo uses a CPU and local `Source_Data`. It refits the small SINDy model and runs small circuit/resource calculations; the FEM simulations, large transport simulations, hardware experiments and full resource sweeps are loaded from their archived results. No quantum service account is needed.

## Setup

Use the supplied local `Source_Data` folder; the Zenodo dataset is not yet published. Place the folder beside this repository, or set `KVN_SOURCE_DATA` before starting the notebook. New example outputs go to `Source_Data/Generated_Outputs/Demo` (or `KVN_DEMO_OUTPUT`); archived inputs are not changed.

In [ ]:
from pathlib import Path
import json
import os
import sys

# Set thread counts before importing numerical libraries.
for variable in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(variable, "1")
ROOT = Path.cwd()
if not (ROOT / "Source").is_dir() and (ROOT / "Codes/Source").is_dir():
    ROOT = ROOT / "Codes"
if not (ROOT / "Source").is_dir():
    raise RuntimeError("Open the notebook from the repository directory containing Source/.")
sys.path.insert(0, str(ROOT))
# Require local data explicitly; this demo does not initiate downloads.
data = Path(os.environ.get("KVN_SOURCE_DATA", ROOT.parent / "Source_Data")).expanduser()
if not (data / "manifest.csv").is_file():
    raise FileNotFoundError("Use the supplied local Source_Data folder or set KVN_SOURCE_DATA to it. The Zenodo dataset is not yet published; this demo does not download data. See README.md.")
from Source.data_locations import set_source_data_root, source_file, shared_data, figure_data
DATA = set_source_data_root(data)
OUT = Path(os.environ.get("KVN_DEMO_OUTPUT", DATA / "Generated_Outputs/Demo"))
OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
from PIL import Image as PILImage
from io import BytesIO

%matplotlib inline
from Figure_Plots.diagnostic_style import diagnostic_style, COLORS, TIME_LABEL

def show_figure(number, width=900):
    """Display the bundled figure preview; numerical examples read Source_Data."""
    names = {1: "Figure_1.png", 2: "Figure_2.png", 3: "Figure3.png", 4: "Figure_4.png", 5: "Figure5_updated.png"}
    path = DATA / "Additional_Source_Data/Demo_Previews" / f"F{number}" / names[number]
    if not path.is_file():
        raise FileNotFoundError(f"Restore Figure {number} preview under Source_Data/Additional_Source_Data/Demo_Previews/.")
    with PILImage.open(path) as picture:
        if number == 2:
            # The original artwork has transparency; display it on white in dark themes.
            fig, ax = plt.subplots(figsize=(width / 100, width / 100 * picture.height / picture.width),
                                   facecolor="white")
            ax.imshow(picture)
            ax.set_axis_off()
            fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
            plt.show()
            return
        picture.thumbnail((1400, 2800))
        buffer = BytesIO()
        picture.save(buffer, format="PNG")
    display(Image(data=buffer.getvalue(), width=width))

## Figure 1 — From wake dynamics to physical resources

**a**, Uncertainty spreads near an unstable state and contracts near a stable state. **b**, CFD snapshots yield POD coordinates and a polynomial SINDy model. **c**, Walsh coefficients encode spatially varying rotation angles. **d**, phase and radial substeps form the transport circuit. **e**, measured modal moments reconstruct the velocity field. **f**, circuit counts, sampling and hardware assumptions determine physical resources.

In [ ]:
show_figure(1)

### a–b. Reduce the flow and identify its dynamics

At $Re=U_\infty D/\nu=100$, two oscillatory modes describe shedding and a shift mode describes the changing mean flow. The steady field and these three modes define the spatial representation. Here we use the archived projected FEM coordinates and **refit** the reflection-symmetric quadratic SINDy model.

`training_coordinates` has shape `(time, 3)` and `training_times` contains uniformly spaced nondimensional times $t^*=tU_\infty/D$. The fit rescales the coordinates by their training RMS; `physical_C` converts the coefficients back to the input coordinate units. To fit another compatible dataset, replace these two arrays. The current fitter assumes three coordinates with reflection $(x,y,z)\mapsto(-x,-y,z)$; it is not an unrestricted SINDy library.

The plots show the **archived FEM projections in their original coordinate units**: x/y oscillations approach sustained shedding, while z tracks the mean-flow shift. They are inputs, not predictions from the refitted model. The printed count sums nonzero coefficients across the three equations; the coefficient difference checks reproduction of the archived fit, not prediction accuracy.


In [ ]:
from Source.POD_SINDy.fit import fit

with np.load(source_file("Data/Model/baseline_inputs.npz")) as archive:
    training_times = archive["training_times"].copy()
    training_coordinates = archive["training_coordinates"].copy()
    projected_times = archive["all_times"].copy()
    projected_coordinates = archive["all_projected_coordinates"].copy()

# Editable model-selection parameters.
degree = 2
threshold = 0.03
fitted = fit(training_coordinates, training_times, degree, threshold)
print(f"Training samples: {len(training_times)}; retained coefficients: {fitted['active_terms']}")
with np.load(source_file("Data/Model/candidate_K2.npz")) as reference:
    if degree == 2 and threshold == 0.03:
        difference = np.max(np.abs(fitted["physical_C"] - reference["physical_coefficients"]))
        print(f"Maximum coefficient difference from the archived fit: {difference:.2e}")
        assert difference < 1e-12

with diagnostic_style():
    fig, axes = plt.subplots(2, 1, figsize=(8, 4.8), sharex=True)
    for j, (name, style) in enumerate(zip(("x", "y"), ("-", "--"))):
        axes[0].plot(projected_times, projected_coordinates[:, j], linestyle=style, label=name)
    axes[1].plot(projected_times, projected_coordinates[:, 2], color=COLORS[2], label="z")
    axes[0].set(ylabel="Raw modal coordinates x, y")
    axes[1].set(xlabel=TIME_LABEL, ylabel="Raw modal coordinate z")
    for ax in axes:
        ax.legend(loc="upper left")
    fig.tight_layout()
    plt.show()


### c. Encode an angle table with Walsh coefficients

On a 16-cell grid, one even layer rotates eight neighbouring cell pairs. Their index is $q=b_0+2b_1+4b_2$, with bits $b_i\in\{0,1\}$. The **left plot** gives each pair's angle, $\vartheta(q)=0.1(2q+1)$ radians: the Figure 1c example scaled by 0.1.

The **right plot** represents the same angles as a sum of Walsh sign patterns. Each bit supplies a sign $(-1)^{b_i}$, the eigenvalue of its Pauli $Z_i$. A mask's binary digits select which signs to multiply: mask 0 is constant ($I$); masks 1, 2 and 4 select bits 0, 1 and 2; mask 3 selects bits 0 and 1. Here

$$\vartheta(q)=0.8-0.1(-1)^{b_0}-0.2(-1)^{b_1}-0.4(-1)^{b_2}.$$

Only masks **0, 1, 2 and 4** are nonzero. Four signed coefficients exactly reproduce all eight angles; the multi-bit terms vanish for this linear table. Negative coefficients contribute signed angles, not probabilities.

`controlled_ry` produces four RZ rotations and six CX gates for this example. The reported `native_T=0` is **before RZ-to-T synthesis**; the arbitrary rotations still incur T-gate cost. `Clifford` includes CX, and `by_gate` gives the detailed breakdown. Change `angles` to encode a finite real table of length $2^{\text{number of controls}}$.

On a uniform grid of $N_x=2^n$ cells, a degree-$K$ polynomial angle table has Walsh terms involving at most $K$ pair-index bits. One interior layer therefore needs at most $\sum_{j=0}^{\min(K,n-1)}\binom{n-1}{j}$ rotations, or $O((\log_2 N_x)^K)$ at fixed degree. This is a rotation-count bound; the complete task also includes time steps, preparation, sampling and error correction.


In [ ]:
from Source.KvN.sparse_walsh import walsh, controlled_ry
from Source.KvN.one_clean_synthesis import expand, inventory

pair_index = np.arange(8)
angles = 0.1 * (2 * pair_index + 1)  # radians; illustrative linear profile
coefficients = walsh(angles)
gates = controlled_ry(angles, controls=[0, 1, 2], target=3)
print("Nonzero Walsh coefficients:", np.count_nonzero(np.abs(coefficients) > 1e-14))
print("Gate inventory:", inventory(expand(gates)))
with diagnostic_style():
    fig, axes = plt.subplots(1, 2)
    axes[0].plot(pair_index, angles, "o-")
    axes[0].set(xlabel="Pair index", ylabel="Rotation angle (rad)")
    markerline, stemlines, baseline = axes[1].stem(pair_index, coefficients)
    baseline.set(color="0.45", linewidth=.8)
    for ax in axes:
        ax.set_xticks(pair_index)
    axes[1].set(xlabel="Walsh mask", ylabel="Coefficient (rad)")
    fig.tight_layout()
    plt.show()


### d–f. Propagate, read out and estimate costs

**d.** A symmetric step places phase half steps around even–odd–even radial rotations. The phase-averaged wake model obeys $\dot r=\mu(r-r^3)$ and $\dot\theta=\omega+\gamma r^2$; the paper reports rounded values $\mu=0.0925$, $\omega=0.753$ and $\gamma=0.302$. Numerical reproduction uses the full-precision archived parameters.

**e.** Radius and phase measurements give mean coordinates for $\langle\mathbf u\rangle=\mathbf u_{st}+\langle x\rangle\boldsymbol\phi_x+\langle y\rangle\boldsymbol\phi_y+\langle z\rangle\boldsymbol\phi_z$. Restoring the known common phase is part of the readout.

**f.** Resource estimation combines a compiled circuit inventory with a required number of accepted samples. Figures 2–3 show that calculation; Figure 4 compares the reconstructed flow with FEM.

For full FEM replay, run `python -m Source.CFD.reproduce_cfd_forces --verify` separately. To rebuild the spatial representation, run `python -m Source.POD_SINDy.replay_frozen_basis --out-dir ../Source_Data/Generated_Outputs/Replayed_POD`. These calculations are not triggered by this notebook.

## Figure 2 — Architecture and resource selection

**a**, A surface-code core consumes T states supplied by a magic-state factory. **b**, the slower of core consumption and factory supply determines the effective T-state interval. **c**, architecture and rotation-synthesis precision are selected jointly within the per-attempt error allowance.

In [ ]:
show_figure(2, width=1000)

### a–b. Specify a circuit and its hardware assumptions

The example JSON starts from the 33-step radial inventory, but is separate from the archived workload. Replace its numbers with those of your own circuit:

- `RZ`: arbitrary-angle Z rotations **per attempt**.
- `native_T`: T/T-dagger gates **before** RZ synthesis.
- `logical_qubits`: total circuit width, including auxiliary qubits.
- `successful_shots`: required accepted samples; `success_probability`: acceptance probability per attempt.
- `source_angle_error_bound`: any existing angle-rounding error bound; use zero only when justified.

The total synthesized count is $N_T=N_T^{native}+N_{R_Z}n_T$, where $n_T$ is the T cost of one RZ rotation. Task time is $N_T\tau_T S/P_s$: $\tau_T$ is the effective T-state interval, $S$ is the required accepted-shot count, and $P_s$ is the circuit acceptance probability. Thus $S/P_s$ is the expected number of attempts. Preparation and distillation acceptance are already included in the factory supply interval; do not apply them again as `success_probability`.

This example uses physical error probability $5\times10^{-4}$, a 1 μs surface-code round, 10 μs classical reaction time and per-attempt error allowance $10^{-4}$. It requires matching preparation data and the recorded preparation-footprint table from Source Data. A custom `--prep-dir` replaces the preparation CSVs, not that footprint calibration. The general CLI's `--p 5` denotes $5\times10^{-4}$, not 5%.

The next cell saves `my_workload.json` and loads it for the search; it does not yet calculate a hardware design.


In [ ]:
from Source.RE.preparation import load_workloads
from Source.RE.optimization import select_designs

custom = json.loads((DATA / "Additional_Source_Data/Example_Inputs/resource_workload.json").read_text())
# Edit these values to describe your circuit.
custom["my_circuit"].update(RZ=4803, native_T=7392, logical_qubits=9,
                            successful_shots=815, success_probability=1.0)
custom_path = OUT / "my_workload.json"
custom_path.write_text(json.dumps(custom, indent=2) + "\n")
workload = load_workloads(custom_path)["my_circuit"]

### c. Run a small design search

For a short demo, search only the fast data block and odd core distances 13–21, with at most one million physical qubits. The results are optima **within this restricted search**, not a general optimum for an arbitrary circuit. A circuit with no feasible design in this range needs a wider search or a different budget.

Each table row has a different objective: `minimum_qubits` minimizes physical qubits; `fastest_under_cap` minimizes task time below the cap. `d_core` is the surface-code distance, “T per RZ” is the chosen synthesis cost, and “Total T per attempt” includes native and synthesized T gates. `task_hours` estimates hardware runtime for all accepted samples and retries, not notebook execution time.

For the default 815-sample task, the minimum-qubit design uses 62,182 physical qubits and about 19.84 hours; the faster design uses 559,953 qubits and 1.62 hours. More hardware reduces runtime within the same error allowance. Changing the circuit requires recalculating the design.


In [ ]:
minimum, fastest, _, _, _ = select_designs(
    "my_circuit", 5e-4, shared_data("RE_common", "RE", "prep_p5.csv"), workload,
    blocks=("fast",), d_min=13, d_max=21, Q_cap=1_000_000, uncapped=False,
)
resource_results = pd.DataFrame([minimum, fastest])
resource_results["task_hours"] = resource_results["complete_task_seconds"] / 3600
resource_results.to_csv(OUT / "resource_estimates.csv", index=False)
display(resource_results[["selection", "physical_qubits", "d_core", "N_T", "T_count", "task_hours"]]
        .rename(columns={"N_T": "T per RZ", "T_count": "Total T per attempt"}).round(3))

Run the saved workload from a terminal opened in `Codes/`:

```bash
python -m Source.RE.run --workloads "${KVN_DEMO_OUTPUT:-${KVN_SOURCE_DATA:-../Source_Data}/Generated_Outputs/Demo}/my_workload.json" --p 5 \
  --blocks fast --d-min 13 --d-max 21 --capped-only \
  --output "${KVN_SOURCE_DATA:-../Source_Data}/Generated_Outputs/Custom_RE"
```

Use the same `KVN_SOURCE_DATA` and `KVN_DEMO_OUTPUT` values as the notebook. The workload is saved at `custom_path`; if the notebook and terminal use different environments, substitute that exact path for `--workloads`. The example above uses Bash/zsh syntax.

For the standard search, omit `--blocks`, `--d-min`, `--d-max` and `--capped-only`. It searches all three data blocks and also seeks the fastest design without a physical-qubit cap. This takes longer. Hardware round/reaction times remain model assumptions; the CLI does not expose them as JSON workload fields.


## Figure 3 — Physical qubits and task time

**a**, Hardware milestones place the estimates in the early fault-tolerant window: $10^4$–$10^6$ physical qubits at physical error probabilities $10^{-4}$–$10^{-3}$. **b**, Open and filled symbols mark minimum-qubit and fastest designs on the physical-qubit/error-probability axes. **c**, Bars give total task time, including accepted samples and retries; the inset gives time per accepted shot. Axis breaks separate very different time scales.

The wake comparison uses the same field-error target for Walsh–KvN and BE–KvN. LCHS–KvN is a separate sine-drift benchmark assigned 815 accepted shots; it is not a matched wake simulation. This figure uses the bundled publication preview; numerical records are read below.


In [ ]:
show_figure(3, width=760)

### b–c. Inspect the numerical designs

The following values come from the full archived search at physical error probability $5\times10^{-4}$. The fastest radial design takes about 1.62 hours on 559,953 physical qubits; the minimum-qubit design uses 53,382 qubits and about 20.07 hours. Both target 815 accepted samples.

The full search also includes compact and intermediate data blocks, explaining the smaller minimum-qubit count than in the fast-block-only demo. `fastest_uncapped` removes the demo's qubit cap; its selected radial design happens to remain below one million qubits.


In [ ]:
selected = pd.read_csv(shared_data("RE_common", "RE", "reference_selected.csv"))
radial = selected[(selected["model"] == "direct") & np.isclose(selected["p_phys"], 5e-4)].copy()
radial["task_hours"] = radial["complete_task_seconds"] / 3600
display(radial[["selection", "physical_qubits", "successful_shots", "task_hours"]].round(3))
assert set(radial["physical_qubits"]) == {53382, 559953}

## Figure 4 — Wake reconstruction and model selection

**a**, Reconstructed transverse velocity is compared with FEM. **b**, validation error is balanced against the rotation bound when selecting polynomial degree. **c–d**, modal trajectories and shift describe growth and saturation. **e**, different reference comparisons separate model error from circuit bias. **f**, drag and lift test derived physical observables.

In [ ]:
show_figure(4, width=880)

### a–b. Compare fields and model complexity

Panel a uses the saved FEM and circuit-reconstructed fields. Panel b selects a quadratic model from the degree/threshold candidates. The model was refitted above; the following cell reads archived cross-validation results without repeating the folds.

Validation is against the **projected FEM field**. The cubic model reduces mean validation error from 5.62% to 2.85%, but increases the rotation bound from 226 to 1,297 for one interior layer in each of three 64-cell coordinates. The paper adopts the quadratic model to balance these costs. The plot below shows validation error only; Figure 4b shows the rotation-cost tradeoff.

Each point is the selected candidate for that library degree, averaged over held-out time-block tests; lower is better. The final printed line describes the model fitted earlier, independently of this archived comparison plot.


In [ ]:
degree_results = pd.read_csv(figure_data("F4") / "model_degree/selected_by_library.csv")
with diagnostic_style():
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.plot(degree_results["library_degree"], degree_results["mean_validation_field_error_percent"], "o-")
    ax.set(xlabel="Polynomial library degree", ylabel="Mean validation field error (%)",
           xticks=degree_results["library_degree"])
    adopted = degree_results.loc[degree_results["library_degree"] == 2].iloc[0]
    ax.annotate("Adopted model", xy=(2, adopted["mean_validation_field_error_percent"]),
                xytext=(16, 16), textcoords="offset points",
                arrowprops={"arrowstyle": "->", "color": "0.3"})
    fig.tight_layout()
    plt.show()
print(f"Refitted degree {degree} model: {fitted['active_terms']} active coefficients at threshold {threshold}.")

### c–e. Distinguish model differences from circuit bias

Panel c compares the FEM trajectory with ideal circuit means in scaled modal coordinates; the earlier raw FEM traces use the original coordinate units. Panel d tests the radial closure $z\approx r^2$. For the circuit ensemble, it compares $\langle z\rangle$ with $\langle r^2\rangle$, which need not equal $\langle x\rangle^2+\langle y\rangle^2$.

In panel e, grey compares the three-coordinate SINDy field with FEM, and red compares the radial closure with the SINDy field; both use the FEM field norm. Blue instead compares the circuit ensemble with the continuum radial model. The 1% target applies to this latter field task, including sampling and implementation error; it is not a claim of 1% agreement with FEM.

For a full recalculation of the model-error curves, use `python -m Source.ROM.generate_fig4_error_data` with a writable copy of Source Data, as described in the README.


### f. Read out drag and lift

$C_D=F_x/(\tfrac12\rho U_\infty^2D)$ and $C_L=F_y/(\tfrac12\rho U_\infty^2D)$ are nondimensional drag and lift coefficients per unit span. The FEM reference integrates pressure and viscous traction; the reduced model uses the calibrated force observables.

Here the relative RMS discrepancy is

$$\varepsilon_q=100\sqrt{\frac{\sum_i(q_i-q_i^{FEM})^2}{\sum_i(q_i^{FEM})^2}},\qquad q\in\{C_D,C_L\}.$$

The force observables were calibrated on FEM data over $10\le t^*\le60$. This comparison uses 34 circuit times within that window, with FEM interpolated to those times. The archived ideal expectations give approximately **0.19% drag and 8.81% lift discrepancy**. The main-text values, 0.21% and 10.5%, additionally include sampling variance for 815 ideal shots per time.

Below, drag is on the left and lift on the right. Dashed curves are interpolated FEM; solid curves are ideal circuit expectations, without Figure 4f's finite-shot points. The printed errors compare the sampled time histories, so both amplitude and phase differences contribute.


In [ ]:
fem = pd.read_csv(figure_data("F4") / "plot_values/Re100/fig4f_fem_forces.csv")
force = pd.read_csv(figure_data("F4") / "plot_values/Re100/fig4f_walsh_forces.csv")
force_errors = {}
with diagnostic_style():
    fig, axes = plt.subplots(1, 2, figsize=(9, 3))
    for ax, name in zip(axes, ("C_D", "C_L")):
        reference = np.interp(force["time"], fem["time"], fem[name])
        force_errors[name] = 100 * np.linalg.norm(force[name] - reference) / np.linalg.norm(reference)
        ax.plot(force["time"], reference, "k--", label="FEM")
        ax.plot(force["time"], force[name], color=COLORS[0], label="Ideal circuit readout")
        ax.set(xlabel=TIME_LABEL, ylabel=f"${name}$")
    axes[0].legend(fontsize=8)
    fig.tight_layout()
    plt.show()
for name, error in force_errors.items():
    print(f"{name}: relative RMS discrepancy = {error:.3f}%")
archived_errors = pd.read_csv(DATA / "Figures/Fig_4/force_sampling/force_error_scales.csv").set_index("observable")
for name, error in force_errors.items():
    np.testing.assert_allclose(error, archived_errors.loc[name, "noiseless_relative_RMS_percent"], rtol=1e-10)

## Figure 5 — Higher degrees and three-coordinate dynamics

**a**, Scalar and coupled polynomial benchmarks test moment accuracy across initial states. Diamonds show shot-noise-free errors; squares and bars show medians and 2.5th–97.5th percentiles of 5,000 estimates with 10,000 shots each. Colors identify initial-state locations. **b**, Quintic transport compares the initial density (grey) with final circuit (blue) and classical (dashed) densities. **c**, The coupled cubic model has its own resource estimate. **d**, Growth and late-time circuit means are compared with dashed continuum-ensemble references. **e**, Three-coordinate wake costs depend on the initial window and grid.

In c and e, minimum-qubit and fastest designs trade hardware for task time; grey shading marks hardware outside the early fault-tolerant window. These are archived simulation and sampling results, not new hardware measurements.


In [ ]:
show_figure(5, width=980)

### a–b. Prepare a compact state

MPS compression controls initial-state preparation cost. This example changes the bond dimension of a real 64-cell Gaussian amplitude, formed as the square root of its normalized probability distribution. It demonstrates the preparation API; it does not rerun the quintic propagation in panel b.

Bond dimension limits the correlations retained between register partitions. The printed fidelity is $F=|\langle\psi_{target}|\psi_{MPS}\rangle|^2$: values closer to one mean less preparation error, generally at higher preparation cost. A displayed 1.00000000 is rounded, not proof of exact equality.

`compress` expects a real normalized vector of power-of-two length and a supported power-of-two bond dimension. General complex amplitudes and arbitrary register layouts are outside this example.


In [ ]:
from Source.KvN.mps import compress

centres = (np.arange(64) + 0.5) / 64
probability = np.exp(-0.5 * ((centres - 0.4) / 0.08)**2)
probability /= probability.sum()
target_amplitude = np.sqrt(probability)
for bond_dimension in (2, 4, 8):
    _, _, _, diagnostics = compress(target_amplitude, bond_dimension)
    print(f"Bond dimension {bond_dimension}: fidelity = {diagnostics['fidelity']:.8f}")

### c–e. Choose the required number of samples

The coupled cubic and three-coordinate wake results use distinct workloads. For the wake, the late-time circuit uses 100 accepted samples; the growth circuit uses 437 at the 1% field target. Their fastest estimates in Table 1 are approximately 4.14 hours and 11.87 days, respectively. Both wake circuits use second-order centred spatial differences.

`field_budget` takes the coordinate grid, mean, reference mean, covariance and a field metric mapping coordinate error to relative velocity-field error. Its conservative expected-RMS bound has the form $B^2+V/S\le\epsilon^2$: $B$ bounds deterministic plus implementation bias, $V$ bounds per-shot field variance, $S$ is the accepted-shot count and $\epsilon$ is the target. More shots reduce the variance term, not the bias bound. This bounds the expected squared error, not every measurement realization.

Here 1% requires 100 accepted shots and 0.5% requires 442. “Not attainable by more shots” at 0.1% means this bound cannot certify the target for the fixed input and allowance; it is not proof that the true error must exceed it.

Change `targets` or `implementation_allowance` while keeping the archived late-time distribution fixed. The latter is a total-variation allowance, not a relative-field percentage. For a new distribution, replace its moments, covariance and field metric consistently.


In [ ]:
from Source.ROM.field_budget import field_budget

readout = json.loads((figure_data("F5") / "wake/data/restart40/terminal_readout.json").read_text())
targets = [0.001, 0.005, 0.01, 0.02, 0.05]  # fractions: 0.01 means 1%
implementation_allowance = 1e-4
budgets = []
for target in targets:
    result = field_budget(readout, implementation_allowance, target)
    shots = result["minimum_accepted_shots_under_rms_bound"]
    budgets.append({"target_percent": 100*target, "accepted_shots": shots})
    description = "not attainable by more shots" if shots is None else f"{shots} accepted shots"
    print(f"Target {100*target:g}%: {description}")
pd.DataFrame(budgets).to_csv(OUT / "measurement_budgets.csv", index=False)

## Run the complete calculations

This demo keeps the calculations small and reads the expensive reference results. The [README](README.md) provides commands for full figure reproduction, fresh scientific calculations, the Linux container and the input formats used above.

- `python -m Figure_Plots.run all`: redraw released figures from archived numerical inputs.
- `python -m Tests.windows_cpp_smoke` and `python -m Source.RE.export_table1 --check`: run public native-kernel and table checks.
- `python -m Source.Data_Production.end_to_end --source-data <working-copy>`: regenerate results from the retained scientific inputs; includes expensive stages and external QuCF inputs.

The archived hardware measurements can be analyzed offline. Running a hardware acquisition script is a separate experiment and is not part of this demo.